# Oil Futures Scenario Forecasting — v0.4 validation

This notebook produces the headline result of the project: a **balanced rolling
pseudo-out-of-sample ablation** of joint WTI/Brent scenario generators, with
statistical significance and formal tail backtests.

What changed in v0.4 relative to v0.3:

| Issue in v0.3 | Fix |
|---|---|
| A model that failed at an origin was silently skipped, so models were scored on different dates | An origin is committed only if **every** model produced a forecast; drops are logged |
| Each model drew its own random numbers | One uniform array per origin, shared by all models (**common random numbers**) |
| No measure of uncertainty on the CRPS leaderboard | **Diebold–Mariano** with Newey–West HAC + HLN correction, plus block-bootstrap intervals |
| No external benchmark | **Random walk** and **filtered historical simulation (AR–GARCH-t)** added |
| `VECM_equal` missing from the grid | Full 2×3 design |
| Re-centering assumed | Ablated via `AR_macro_nc` |
| Tail calibration only as a hit rate | **Kupiec**, **Christoffersen** and **Acerbi–Székely Z2** |
| Live yfinance calls | On-disk parquet cache |

Run `python scripts/fetch_data.py` once before this notebook.

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from oil_futures_regime import (
    DEFAULT_MODEL_SPECS,
    aggregate_oos, best_model_summary, build_macro_features, common_uniforms,
    fit_fhs_model, fit_joint_ar_model, fit_rw_model, fit_vecm_scenario_model,
    load_or_download, paired_score_table, pit_summary, prepare_residual_pool,
    roll_gap_diagnostics, rolling_oos_validate, simulate, var_es_report,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

In [ ]:
START, END, ANCHOR = "2013-01-01", "2025-03-08", "W-FRI"   # yfinance end is exclusive
HORIZONS = [1, 4, 12, 20]
H_MAX = max(HORIZONS)

HALF_LIFE = 52
MIN_MACRO_ESS_FRACTION = 0.35     # fraction of the pool, not an absolute count
N_SIM_ILLUSTRATIVE = 10_000
N_SIM_OOS = 2_000
OOS_START = "2018-01-01"
STRIDE = 4                        # roughly monthly origins
BASELINE = "RW_equal"
SEED = 2026

CACHE = ROOT / "data" / "cache"
OUT = ROOT / "reports" / "oos"
OUT.mkdir(parents=True, exist_ok=True)

## 2. Data

Everything is read from the parquet cache written by `scripts/fetch_data.py`, so
this notebook is deterministic and does not depend on what Yahoo returns today.

In [ ]:
oil = load_or_download("oil", {"WTI": "CL=F", "BRENT": "BZ=F"},
                       START, END, ANCHOR, cache_dir=CACHE, dropna="any")
state = load_or_download("state", {"VIX": "^VIX", "DXY": "DX-Y.NYB", "TNX": "^TNX"},
                         START, END, ANCHOR, cache_dir=CACHE, dropna="all")

logp = np.log(oil[["WTI", "BRENT"]]).dropna()
macro = build_macro_features(state)

print(f"{len(logp)} weekly observations, {logp.index.min().date()} -> {logp.index.max().date()}")
display(logp.tail(3))
display(macro.tail(3))

### Data-quality caveat: front-month splicing

`CL=F` and `BZ=F` are **concatenated front-month** series, not roll-adjusted
ones. Weekly returns spanning a contract roll therefore contain a price jump
that is not a tradeable return, and those jumps contaminate the spread that the
VECM models. The diagnostic below quantifies how large the problem is instead of
leaving it as a sentence in the limitations section.

In [ ]:
flags = roll_gap_diagnostics(oil, z_threshold=4.0)
print(f"{len(flags)} weekly moves beyond 4 robust sigma")
display(flags.head(10))

## 3. One transparent forecast origin

Before the rolling harness, a single origin makes the mechanics visible. This is
an illustration, **not** evidence: one origin cannot rank models.

In [ ]:
origin_pos = len(logp) - 1 - H_MAX
origin = logp.index[origin_pos]
train = logp.iloc[: origin_pos + 1]

models = {
    "RW": fit_rw_model(train),
    "AR": fit_joint_ar_model(train, p_max=3),
    "VECM": fit_vecm_scenario_model(train, coint_rank=1, k_ar_diff=1, deterministic="ci"),
    "FHS": fit_fhs_model(train, p_max=3),
}
print("Origin:", origin.date(), " target:", logp.index[origin_pos + H_MAX].date())
print("VECM beta:")
display(pd.DataFrame(models["VECM"].result.beta, index=models["VECM"].columns, columns=["beta"]))
for name, m in models.items():
    print(f"{name:5s} residual pool {m.residuals.shape}")

In [ ]:
# One uniform array shared by every model: differences below are model differences.
u = common_uniforms(H_MAX, N_SIM_ILLUSTRATIVE, seed=SEED)

paths, diags = {}, {}
for family, mode in [("RW", "equal"), ("AR", "time"), ("AR", "macro"),
                     ("VECM", "time"), ("VECM", "macro"), ("FHS", "time")]:
    m = models[family]
    pool, prob, d = prepare_residual_pool(
        m.residuals, mode=mode, half_life=HALF_LIFE, macro_features=macro,
        origin=origin, min_macro_ess_fraction=MIN_MACRO_ESS_FRACTION, recenter=True,
    )
    key = f"{family}_{mode}"
    paths[key] = simulate(m, pool, prob, horizon=H_MAX, n_sim=N_SIM_ILLUSTRATIVE, u=u)
    diags[key] = {"ess": d.ess, "ess_fraction": d.ess_fraction,
                  "bandwidth": d.bandwidth, "n_pool": len(pool)}

display(pd.DataFrame(diags).T.round(3))

In [ ]:
rows = []
actual = logp.iloc[origin_pos + H_MAX]
for key, p in paths.items():
    wti, brent = p[H_MAX, :, 0], p[H_MAX, :, 1]
    rows.append({
        "model": key,
        "WTI_q05": np.exp(np.quantile(wti, 0.05)),
        "WTI_q50": np.exp(np.quantile(wti, 0.50)),
        "WTI_q95": np.exp(np.quantile(wti, 0.95)),
        "WTI_actual": float(np.exp(actual["WTI"])),
        "SPREAD_sd": float((wti - brent).std()),
    })
display(pd.DataFrame(rows).round(3))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True)
h = np.arange(H_MAX + 1)
realized = logp.iloc[origin_pos : origin_pos + H_MAX + 1]

for ax, key in zip(axes.ravel(), ["RW_equal", "AR_macro", "VECM_macro", "FHS_time"]):
    p = paths[key]
    for j, (label, sim, real) in enumerate([
        ("WTI", p[:, :, 0], realized["WTI"].values),
        ("SPREAD", p[:, :, 0] - p[:, :, 1], (realized["WTI"] - realized["BRENT"]).values),
    ]):
        if j == 1:
            continue
        q = np.quantile(sim, [0.05, 0.50, 0.95], axis=1)
        ax.fill_between(h, q[0], q[2], alpha=0.2)
        ax.plot(h, q[1], label="median")
        ax.plot(h, real, "k--", label="realized")
    ax.set_title(f"{key} — WTI log price")
    ax.set_xlabel("weeks ahead")
axes[0, 0].legend()
plt.tight_layout(); plt.show()

## 4. Rolling pseudo-OOS ablation

Every model is re-estimated at every origin using data available at or before
that origin: AR order selection, VECM, the GARCH filter, the macro
standardization and the kernel bandwidth. An origin enters the panel only if all
eleven models produced a forecast.

In [ ]:
print([s.name for s in DEFAULT_MODEL_SPECS])

vo = rolling_oos_validate(
    log_prices=logp,
    macro_features=macro,
    horizons=HORIZONS,
    oos_start=OOS_START,
    stride=STRIDE,
    n_sim=N_SIM_OOS,
    half_life=HALF_LIFE,
    min_macro_ess_fraction=MIN_MACRO_ESS_FRACTION,
    model_specs=DEFAULT_MODEL_SPECS,
    seed=SEED,
)

print(f"\nOrigins kept: {len(vo.origins_used)}   dropped: {len(vo.origins_dropped)}")
print(f"Balanced panel: {vo.is_balanced}")
display(vo.failures.head(10) if not vo.failures.empty else "no failures")

In [ ]:
summary = aggregate_oos(vo.results)
for variable in ["WTI", "BRENT", "SPREAD"]:
    print(f"\n{variable} — mean CRPS (lower is better)")
    display(summary[summary["variable"] == variable]
            .pivot(index="model", columns="horizon_weeks", values="mean_crps").round(5))

### The leaderboard on its own is not a result

With weekly data, monthly origins and a 20-week horizon, consecutive forecast
windows overlap almost entirely: the effective number of independent evaluations
at h=20 is roughly `n_origins * stride / h`, i.e. under 20. A 2–3% CRPS gap is
well inside the noise band. The table below attaches a Diebold–Mariano test
(HAC variance at the implied overlap lag, HLN small-sample correction) and a
circular block-bootstrap interval to every comparison against the baseline.

In [ ]:
sig = paired_score_table(vo.results, baseline=BASELINE, stride=STRIDE)
best = best_model_summary(sig)
display(best.round(4))

winners = sig[(sig["significant_5pct"]) & (sig["improvement_pct"] > 0)]
print("\nSignificant improvements over the baseline at 5%:")
display(winners[["variable", "horizon_weeks", "model", "improvement_pct",
                 "dm_stat", "dm_pvalue", "boot_lo", "boot_hi", "eff_independent_n"]].round(4)
        if not winners.empty else "none")

In [ ]:
# Full comparison for one cell, so the size of the noise band is visible.
cell = sig[(sig["variable"] == "SPREAD") & (sig["horizon_weeks"] == 20)]
display(cell[["model", "mean_crps", "improvement_pct", "dm_stat", "dm_pvalue",
              "boot_lo", "boot_hi", "n_origins", "eff_independent_n"]].round(4))

## 5. Calibration: coverage, width and PIT

In [ ]:
cov = summary.pivot_table(index="model", columns=["variable", "horizon_weeks"],
                          values="coverage_90").round(3)
print("90% interval coverage (target 0.90)")
display(cov)

wid = summary.pivot_table(index="model", columns=["variable", "horizon_weeks"],
                          values="avg_width_90").round(4)
print("90% interval width (sharpness — only meaningful alongside coverage)")
display(wid)

In [ ]:
pit = pit_summary(vo.results)
display(pit.sort_values(["variable", "horizon_weeks", "ks_uniform_stat"]).head(20).round(4))
print("KS p-values are descriptive only: overlapping PIT values are serially dependent.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, model in zip(axes, [BASELINE, "AR_macro", "VECM_macro"]):
    sel = vo.results[(vo.results["model"] == model) &
                     (vo.results["variable"] == "WTI") &
                     (vo.results["horizon_weeks"] == 4)]
    ax.hist(sel["pit"], bins=10, range=(0, 1), edgecolor="white")
    ax.axhline(len(sel) / 10, color="k", ls="--", lw=1)
    ax.set_title(f"PIT — {model}, WTI, h=4")
plt.tight_layout(); plt.show()

## 6. Tail backtests

Coverage rates do not tell you whether a tail is mis-specified. The report below
runs Kupiec (unconditional coverage), Christoffersen (independence and
conditional coverage) and the Acerbi–Székely Z2 severity statistic for expected
shortfall. Because overlapping forecasts violate the independence assumption,
the panel is thinned so retained origins have non-overlapping target windows;
cells with fewer than two expected exceptions are flagged `underpowered` rather
than reported as passes.

In [ ]:
risk = var_es_report(vo.results, stride=STRIDE, alphas=(0.05, 0.01))
r5 = risk[(risk["alpha"] == 0.05) & (~risk["underpowered"])]
display(r5[["variable", "horizon_weeks", "model", "n_used", "exception_rate",
            "kupiec_p", "christoffersen_ind_p", "christoffersen_cc_p",
            "es_z2", "es_z2_p"]].round(4).head(30))

print("\nCells rejected at 5% by conditional coverage:")
display(r5[r5["christoffersen_cc_p"] < 0.05][["variable", "horizon_weeks", "model",
                                              "exception_rate", "christoffersen_cc_p"]].round(4))

## 7. Pre-registered decision rule

Fixed **before** looking at the numbers, so the conclusion is not chosen after
the fact:

1. A component stays only if it beats the relevant comparator with a
   Diebold–Mariano p-value below 0.05 **and** a bootstrap interval that excludes
   zero. Ranking first is not enough.
2. `RW_equal` is the benchmark for levels. If nothing beats a random walk on WTI
   and Brent, that is the finding, and the honest headline is that the value of
   the engine is in the **spread** and in **tail calibration**, not in point-level
   forecasting.
3. Recency weighting stays only if `AR_time` beats `AR_equal`.
4. Macro conditioning stays only if `AR_macro` beats `AR_time` — and it must also
   beat `FHS_time`, since a GARCH filter is the cheap way to capture the same
   conditional-volatility information.
5. Error correction stays if `VECM_*` beats the matching `AR_*`, especially on the
   spread and at longer horizons.
6. Re-centering stays if `AR_macro` beats `AR_macro_nc`.

A negative result is a result: "macro-state conditioning changes the scenario
weights but does not improve density forecasts once recency and a volatility
filter are accounted for" is a defensible, publishable conclusion.

In [ ]:
vo.results.to_csv(OUT / "forecast_rows.csv.gz", index=False, compression="gzip")
summary.to_csv(OUT / "summary.csv", index=False)
sig.to_csv(OUT / "significance.csv", index=False)
best.to_csv(OUT / "best_models.csv", index=False)
pit.to_csv(OUT / "pit.csv", index=False)
risk.to_csv(OUT / "var_es.csv", index=False)
vo.failures.to_csv(OUT / "failures.csv", index=False)
print("written to", OUT)

## 8. Next structural extension

The highest-value follow-up is a **cointegration-consistent state space**: a
latent common stochastic trend plus a stationary relative component, estimated
by Kalman filtering, so the state-space block and the VECM tell the same story
rather than sitting side by side. After that: contract-level futures curves
(roll-adjusted returns, calendar spreads, contango/backwardation), and only then
a decision layer with transaction costs.